In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Write a program that multiplies two matrices of 32-bit floating point numbers on a GPU.
  Given matrix $A$ of dimensions $M \times N$ and matrix $B$ of dimensions $N \times K$, compute
  the product matrix $C = A \times B$, which will have dimensions $M \times K$.
  All matrices are stored in row-major format.
</p>

<h2>Implementation Requirements</h2>
<ul>
  <li>Use only native features (external libraries are not permitted)</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>The final result must be stored in matrix <code>C</code></li>
</ul>

<h2>Example 1:</h2>
<p>
Input:<br>
Matrix $A$ ($2 \times 2$):
$$
\begin{bmatrix}
1.0 & 2.0 \\
3.0 & 4.0
\end{bmatrix}
$$
Matrix $B$ ($2 \times 2$):
$$
\begin{bmatrix}
5.0 & 6.0 \\
7.0 & 8.0
\end{bmatrix}
$$
Output:<br>
Matrix $C$ ($2 \times 2$):
$$
\begin{bmatrix}
19.0 & 22.0 \\
43.0 & 50.0
\end{bmatrix}
$$
</p>

<h2>Example 2:</h2>
<p>
Input:<br>
Matrix $A$ ($1 \times 3$):
$$
\begin{bmatrix}
1.0 & 2.0 & 3.0
\end{bmatrix}
$$
Matrix $B$ ($3 \times 1$):
$$
\begin{bmatrix}
4.0 \\
5.0 \\
6.0
\end{bmatrix}
$$
Output:<br>
Matrix $C$ ($1 \times 1$):
$$
\begin{bmatrix}
32.0
\end{bmatrix}
$$
</p>

<h2>Constraints</h2>
<ul>
  <li>1 &le; <code>M</code>, <code>N</code>, <code>K</code> &le; 8192</li>
  <li>Performance is measured with <code>M</code> = 8192, <code>N</code> = 6144, <code>K</code> = 4096</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

__global__ void matrix_multiplication_kernel(const float* A, const float* B, float* C, int M, int N,
                                             int K) {}

// A, B, C are device pointers (i.e. pointers to memory on the GPU)
extern "C" void solve(const float* A, const float* B, float* C, int M, int N, int K) {
    dim3 threadsPerBlock(16, 16);
    dim3 blocksPerGrid((K + threadsPerBlock.x - 1) / threadsPerBlock.x,
                       (M + threadsPerBlock.y - 1) / threadsPerBlock.y);

    matrix_multiplication_kernel<<<blocksPerGrid, threadsPerBlock>>>(A, B, C, M, N, K);
    cudaDeviceSynchronize();
}


# CUTE

In [ ]:
%%writefile solution_cute.py
import cutlass
import cutlass.cute as cute


# A, B, C are tensors on the GPU
@cute.jit
def solve(
    A: cute.Tensor, B: cute.Tensor, C: cute.Tensor, M: cute.Int32, N: cute.Int32, K: cute.Int32
):
    pass


# JAX

In [ ]:
%%writefile solution_jax.py
import jax
import jax.numpy as jnp


# A, B are tensors on GPU
@jax.jit
def solve(A: jax.Array, B: jax.Array, M: int, N: int, K: int) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


def matrix_multiplication_kernel(
    A: UnsafePointer[Float32, MutExternalOrigin],
    B: UnsafePointer[Float32, MutExternalOrigin],
    C: UnsafePointer[Float32, MutExternalOrigin],
    M: Int32,
    N: Int32,
    K: Int32,
):
    pass


# A, B, C are device pointers (i.e. pointers to memory on the GPU)
@export
def solve(
    A: UnsafePointer[Float32, MutExternalOrigin],
    B: UnsafePointer[Float32, MutExternalOrigin],
    C: UnsafePointer[Float32, MutExternalOrigin],
    M: Int32,
    N: Int32,
    K: Int32,
) raises:
    var BLOCK_SIZE: Int32 = 16
    var ctx = DeviceContext()

    var grid_dim_x = ceildiv(K, BLOCK_SIZE)
    var grid_dim_y = ceildiv(M, BLOCK_SIZE)

    var _kernel = ctx.compile_function[matrix_multiplication_kernel, matrix_multiplication_kernel]()
    ctx.enqueue_function(
        _kernel,
        A,
        B,
        C,
        M,
        N,
        K,
        grid_dim=(grid_dim_x, grid_dim_y),
        block_dim=(BLOCK_SIZE, BLOCK_SIZE),
    )

    ctx.synchronize()


# Torch

In [ ]:
%%writefile solution_pytorch.py
import torch


# A, B, C are tensors on the GPU
def solve(A: torch.Tensor, B: torch.Tensor, C: torch.Tensor, M: int, N: int, K: int):
    pass


# Triton

In [ ]:
%%writefile solution_triton.py
import torch
import triton
import triton.language as tl


@triton.jit
def matrix_multiplication_kernel(
    a, b, c, M, N, K, stride_am, stride_an, stride_bn, stride_bk, stride_cm, stride_ck
):
    pass


# a, b, c are tensors on the GPU
def solve(a: torch.Tensor, b: torch.Tensor, c: torch.Tensor, M: int, N: int, K: int):
    stride_am, stride_an = N, 1
    stride_bn, stride_bk = K, 1
    stride_cm, stride_ck = K, 1

    grid = (M, K)
    matrix_multiplication_kernel[grid](
        a, b, c, M, N, K, stride_am, stride_an, stride_bn, stride_bk, stride_cm, stride_ck
    )


# Evaluate Setup

In [ ]:
# Download required files from GitHub
!mkdir -p core
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/challenge_base.py -O core/challenge_base.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/evaluator.py -O core/evaluator.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/easy/2_matrix_multiplication/challenge.py -O challenge.py

from challenge import Challenge
from core.evaluator import Evaluate

ch = Challenge()


# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch, EVAL_LANG)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
